# Olist E-Commerce - 02 : Cleaning & Preprocessing

**Author:** Diego Ospina | **Dataset:** Brazilian E-Commerce Public Dataset by Olist

> The raw data is almost clean, but it needs **structure** before analysis: useful timestamps must be parsed, zip codes must be treated as strings, category names must be translated, several tables must be aggregated to the order grain, and a couple of genuine inconsistencies must be fixed. This notebook turns the messy raw files into tidy datasets saved in `data/processed/`.

### Pipeline summary
1. Reload every raw file from `data/raw/` (idempotent).
2. Clean the small tables: geolocation (de-duplicate), products (rename typos, translate categories).
3. Parse order timestamps and derive delivery metrics for delivered orders.
4. Aggregate payments and reviews to the order grain.
5. Assemble the **master order-level dataset** and write all outputs to `data/processed/`.

> **Note:** this notebook reproduces exactly the logic of `src/build_processed.py`, so the datasets can be regenerated at any time by running this notebook or that script.

## 1. Setup

We import the libraries, configure the display, and declare the folder paths.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW = os.path.join('..', 'data', 'raw')
PROC = os.path.join('..', 'data', 'processed')
os.makedirs(PROC, exist_ok=True)
print('paths ok')

paths ok


## 2. Load raw data

We read every raw CSV. **Zip-code prefixes are loaded as strings** so we never lose leading zeros (right-padding matters for geography).

In [2]:
def load(name, **kw):
    # Carga una tabla cruda desde data/raw
    return pd.read_csv(os.path.join(RAW, name), **kw)

customers  = load('olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
geolocation= load('olist_geolocation_dataset.csv', dtype={'geolocation_zip_code_prefix': str})
order_items= load('olist_order_items_dataset.csv')
payments   = load('olist_order_payments_dataset.csv')
reviews    = load('olist_order_reviews_dataset.csv')
orders     = load('olist_orders_dataset.csv')
products   = load('olist_products_dataset.csv')
sellers    = load('olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})
categories_tr = load('product_category_name_translation.csv')

raw = {'customers': customers, 'orders': orders, 'order_items': order_items, 'products': products, 'payments': payments, 'reviews': reviews, 'sellers': sellers, 'geolocation': geolocation, 'categories_tr': categories_tr}
for name, df in raw.items():
    print(f"{name:14s} -> {df.shape[0]:>9,} x {df.shape[1]}")

customers      ->    99,441 x 5
orders         ->    99,441 x 8
order_items    ->   112,650 x 7
products       ->    32,951 x 9
payments       ->   103,886 x 5
reviews        ->    99,224 x 7
sellers        ->     3,095 x 4
geolocation    -> 1,000,163 x 5
categories_tr  ->        71 x 2


## 3. Clean the small tables

### 3.1 Geolocation - remove duplicates

`geolocation` contains ~1M rows but only ~19k distinct zip prefixes. The official dataset ships with many duplicate coordinate sheets recorded for the same zip. We keep **one representative point per zip** (median lat/lng) and drop exact duplicates first.

In [3]:
before = len(geolocation)
geo_dedup = geolocation.drop_duplicates()
after_dedup = len(geo_dedup)
print(f'Filas originales: {before:,}')
print(f'Tras eliminar duplicados exactos: {after_dedup:,}')
print(f'Duplicados eliminados: {before - after_dedup:,}')

# Una coordenada representativa por prefijo de zip (mediana)
geo = (geo_dedup.groupby('geolocation_zip_code_prefix', as_index=False)
              .agg(geolocation_lat=('geolocation_lat', 'median'),
                   geolocation_lng=('geolocation_lng', 'median'),
                   geolocation_state=('geolocation_state', 'last')))
print('Zips unicos:', len(geo))

Filas originales: 1,000,163
Tras eliminar duplicados exactos: 738,332
Duplicados eliminados: 261,831
Zips unicos: 19015


### 3.2 Products - fix column typos and translate categories

The product file has two columns with a typo (`lenght` instead of `length`). We rename them. We also handle the category field:
- 610 products have a null category -> labelled `not_specified`.
- 2 category names (`pc_gamer`, `portateis_cozinha...`) are missing from the translation table, so we add manual translations.
- The remaining products use the official Portuguese-to-English translation.

In [4]:
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'})

# Traducciones manuales para categorias ausentes en la tabla oficial
EXTRA_TRANSLATIONS = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['pc_gamer', 'portable_kitchen_appliances']})
cat_tr = pd.concat([categories_tr, EXTRA_TRANSLATIONS], ignore_index=True)

cat_map = dict(zip(cat_tr['product_category_name'], cat_tr['product_category_name_english']))
products['category_pt'] = products['product_category_name']
products['category'] = products['product_category_name'].map(cat_map).fillna('not_specified')
products.loc[products['product_category_name'].isna(), 'category'] = 'not_specified'

print('Productos vacios en category:', products['category'].isna().sum())
print('Ejemplos de traduccion:')
print(products[['product_category_name', 'category']].drop_duplicates().head(6))
print('products:', products.shape)

Productos vacios en category: 0
Ejemplos de traduccion:
   product_category_name             category
0             perfumaria            perfumery
1                  artes                  art
2          esporte_lazer       sports_leisure
3                  bebes                 baby
4  utilidades_domesticas           housewares
5  instrumentos_musicais  musical_instruments
products: (32951, 11)


## 4. Orders - timestamps and delivery metrics

We parse every timestamp column and derive two useful delivery measures **only for delivered orders** (non-delivered orders legitimately have no delivery date):

- `delivery_time_days`: days from purchase to customer delivery.
- `delivery_delay_days`: days late vs the estimated date (positive = late).
- `delivery_status`: `On time` (delay <= 0) or `Late` (delay > 0).

In [5]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

orders['year'] = orders['order_purchase_timestamp'].dt.year
orders['month'] = orders['order_purchase_timestamp'].dt.month
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

delivered = orders['order_status'] == 'delivered'

orders['delivery_time_days'] = np.nan
orders.loc[delivered, 'delivery_time_days'] = (
    (orders.loc[delivered, 'order_delivered_customer_date']
     - orders.loc[delivered, 'order_purchase_timestamp']).dt.total_seconds() / 86400)

orders['delivery_delay_days'] = np.nan
orders.loc[delivered, 'delivery_delay_days'] = (
    (orders.loc[delivered, 'order_delivered_customer_date']
     - orders.loc[delivered, 'order_estimated_delivery_date']).dt.total_seconds() / 86400)

orders['delivery_status'] = np.where(orders['delivery_delay_days'] > 0, 'Late', 'On time')
orders.loc[~delivered, 'delivery_status'] = np.nan

print('orders:', orders.shape)
print()
print('Distribucion de delivery_status:')
print(orders['delivery_status'].value_counts(dropna=False))

orders: (99441, 14)

Distribucion de delivery_status:
delivery_status
On time    88652
Late        7826
NaN         2963
Name: count, dtype: int64


## 5. Aggregate payments and reviews to the order grain

`payments` and `reviews` can hold several rows per order (installments, multiple methods, multiple reviews). We collapse them into **one row per order** so they can be joined into the master frame.

In [6]:
# --- Pagos: agregar por orden ---
pay_agg = (payments.groupby('order_id', as_index=False)
                   .agg(total_paid=('payment_value', 'sum'),
                        n_payments=('order_id', 'count'),
                        n_installments_max=('payment_installments', 'max'),
                        payment_type_main=('payment_type', lambda x: x.value_counts().index[0])))
print('pay_agg:', pay_agg.shape)

# --- Resenas: agregar por orden ---
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
rev_agg = (reviews.groupby('order_id', as_index=False)
                   .agg(review_score_mean=('review_score', 'mean'),
                        review_score_min=('review_score', 'min'),
                        review_score_max=('review_score', 'max'),
                        n_reviews=('order_id', 'count'),
                        has_comment_title=('review_comment_title', lambda s: s.notna().any()),
                        has_comment_message=('review_comment_message', lambda s: s.notna().any())))
print('rev_agg:', rev_agg.shape)

pay_agg: (99440, 5)


rev_agg: (98673, 7)


## 6. Items - enrich and aggregate

`order_items` has the line-level detail. We enrich each line with the English category and compute `total_value` (price + freight), then aggregate to the order grain for the master dataset.

In [7]:
# Enriquecer cada item con la categoria y peso del producto
items = order_items.copy()
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])
items['total_value'] = items['price'] + items['freight_value']
items = items.merge(
    products[['product_id', 'category', 'product_weight_g']].rename(columns={'category': 'item_category'}),
    on='product_id', how='left')
print('items:', items.shape)

# Agregados a nivel de orden
item_agg = (items.groupby('order_id', as_index=False)
                   .agg(n_items=('order_item_id', 'count'),
                        n_products=('product_id', 'nunique'),
                        n_sellers=('seller_id', 'nunique'),
                        product_value=('price', 'sum'),
                        freight_value=('freight_value', 'sum'),
                        order_value=('total_value', 'sum'),
                        n_categories=('item_category', 'nunique')))
print('item_agg:', item_agg.shape)

items: (112650, 10)


item_agg: (98666, 8)


## 7. Assemble the master order-level dataset

We now join everything into `master_orders` at the **order grain**. This becomes the canonical table used by every downstream notebook (EDA, statistics, BI, ML).

In [8]:
master = (orders
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='left')
    .merge(pay_agg, on='order_id', how='left')
    .merge(rev_agg, on='order_id', how='left')
    .merge(item_agg, on='order_id', how='left'))

print('master shape:', master.shape)
print()
print('Columnas:')
print(list(master.columns))

master shape: (99441, 33)

Columnas:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'year', 'month', 'purchase_month', 'delivery_time_days', 'delivery_delay_days', 'delivery_status', 'customer_unique_id', 'customer_state', 'total_paid', 'n_payments', 'n_installments_max', 'payment_type_main', 'review_score_mean', 'review_score_min', 'review_score_max', 'n_reviews', 'has_comment_title', 'has_comment_message', 'n_items', 'n_products', 'n_sellers', 'product_value', 'freight_value', 'order_value', 'n_categories']


## 8. Write cleaned datasets

Finally we persist every cleaned table as **parquet** under `data/processed/`. Parquet keeps dtypes, is fast to read, and compresses well. These files are the input for all later notebooks.

In [9]:
outputs = {
    'olist_master_orders.parquet': master,
    'olist_items.parquet': items,
    'olist_catalog.parquet': products,
    'olist_geolocation_clean.parquet': geo,
    'product_category_name_translation.parquet': cat_tr,
}

for fname, df in outputs.items():
    path = os.path.join(PROC, fname)
    df.to_parquet(path)
    print(f"-> {fname}  ({df.shape[0]:,} x {df.shape[1]})")

print('\nListo: todos los datasets limpios guardados en data/processed/')

-> olist_master_orders.parquet  (99,441 x 33)
-> olist_items.parquet  (112,650 x 10)
-> olist_catalog.parquet  (32,951 x 11)
-> olist_geolocation_clean.parquet  (19,015 x 4)
-> product_category_name_translation.parquet  (73 x 2)

Listo: todos los datasets limpios guardados en data/processed/


## 9. Takeaways

- From 9 messy raw files we produced **5 tidy datasets** in `data/processed/`, with a canonical **master at the order grain** (99,441 rows) used by all later notebooks.
- Explicit decisions were made and documented: zip codes as strings, median point per zip, `not_specified` category, delivery metrics only for delivered orders, two manual category translations.
- The process is **reproducible**: re-running this notebook (or `src/build_processed.py`) regenerates the same outputs.

Next: **`03_eda_visualizations.ipynb`** will explore the cleaned data and save figures to `images/`.